In [ ]:
import os
import time
import torch
import argparse 
import math
from torch import nn
from torch.utils import data
from torchvision import transforms
from tools.utils import *
import torch.optim as optim
import tools.mftnetwork as model
from dataset.scene_dataset import *
import torch.nn.functional as F

kl_loss = torch.nn.KLDivLoss(reduction='mean')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
def class_kl_comput(f, index):
    kl = 0.0
    for i in range(len(index)):
        for j in range(len(index)):
            if i == j: continue
            if not torch.is_tensor(f[index[i]]):
                log_a =F.log_softmax(torch.from_numpy(f[index[i]]),dim=-1)
            else:
                log_a =F.log_softmax(f[index[i]],dim=-1)      
            if not torch.is_tensor(f[index[j]]): 
                softmax_b =F.softmax(torch.from_numpy(f[index[j]]),
                                      dim=-1)
            else:
                softmax_b =F.softmax(f[index[j]],dim=-1)              
            kl += kl_loss(log_a,softmax_b)
    count = math.comb(len(index),2)
    kl = kl/(2*count)
    return (torch.from_numpy(kl) if not torch.is_tensor(kl) else kl)

def kl_divergence(p_logits, q_logits, eps=1e-10):
    p = F.softmax(p_logits.squeeze(), dim=0)  
    q = F.softmax(q_logits.squeeze(), dim=0)  
    kl = torch.sum(p * (torch.log(p + eps) - torch.log(q + eps)))
    return kl

def compute_kl_for_index(dataset, index1, index2):
    p = dataset[index1]  
    q = dataset[index2]  
    return kl_divergence(p, q)

def class_kl_comput_new(f, index):
    #if 
    kl = 0.0
    for i in range(len(index)):
        for j in range(len(index)):
            if i == j: continue
            p = f[index[i]]
            q = f[index[j]]         
            kl += kl_divergence(p, q)
    count = math.comb(len(index),2)
    kl = kl/(2*count)
    return (torch.from_numpy(kl) if not torch.is_tensor(kl) else kl)



def model_nas_count(model_nas):
    model_nas = np.round(model_nas)
    model_nas= model_nas.astype(int)
    return model_nas

In [ ]:
def get_data(dataID,print_per_batches,save_path,network,crop_size,root_dir,train_batch_size,val_batch_size,num_workers):
    if dataID==1:
        DataName = 'UCM'
        num_classes = 21
        classname = ('agricultural','airplane','baseballdiamond',
                        'beach','buildings','chaparral',
                        'denseresidential','forest','freeway',
                        'golfcourse','harbor','intersection',
                        'mediumresidential','mobilehomepark','overpass',
                        'parkinglot','river','runway',
                        'sparseresidential','storagetanks','tenniscourt')

    elif dataID==2:        
        DataName = 'AID'
        num_classes = 30
        classname = ('airport','bareland','baseballfield',
                        'beach','bridge','center',
                        'church','commercial','denseresidential',
                        'desert','farmland','forest',
                        'industrial','meadow','mediumresidential',
                        'mountain','parking','park',
                        'playground','pond','port',
                        'railwaystation','resort','river',
                        'school','sparseresidential','square',
                        'stadium','storagetanks','viaduct')
    elif dataID==3:        
        DataName = 'FUSAR'
        num_classes = 4
        classname = ('CargoShip','Fishing','Tanker','BulkCarrier')   
        
    elif dataID==4:        
        DataName = 'OpenSARShip'
        num_classes = 2
        classname = ('Cargo','Tanker')                 
    print_per_batches = print_per_batches
    save_path_prefix = save_path + DataName+'/Pretrain/'+network+'/'
    
    if os.path.exists(save_path_prefix)==False:
        os.makedirs(save_path_prefix)

    composed_transforms = transforms.Compose([
            transforms.Resize(size=(crop_size,crop_size)),            
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.05424311,0.05424311,0.05424311), std=(0.05708374,0.05708374,0.05708374))])
    
    train_loader = data.DataLoader(
        scene_dataset(root_dir=root_dir,pathfile='./dataset/'+DataName+'_train.txt', transform=composed_transforms),
        batch_size=train_batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = data.DataLoader(
        scene_dataset(root_dir=root_dir,pathfile='./dataset/'+DataName+'_test.txt', transform=composed_transforms),
        batch_size=val_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return DataName,num_classes,classname,print_per_batches,train_loader, val_loader,composed_transforms,save_path_prefix

In [ ]:
dataID = 4
print_per_batches = 5
save_path = './'
network = 'fnas'
crop_size = 32
root_dir = './data/'
train_batch_size = 24
val_batch_size = 64
num_workers = 0

num_epochs = 20
lr = 0.0001

In [ ]:
(DataName,
 num_classes,
 classname,
 print_per_batches,
 train_loader, 
 val_loader,
 composed_transforms,
 save_path_prefix) = get_data(dataID,
                                print_per_batches,
                                save_path,
                                network,
                                crop_size,
                                root_dir,
                                train_batch_size,
                                val_batch_size,
                                num_workers)

In [ ]:
# Validating the model architecture
C=4
Sgenotype =  [0,0,1,0,2,1,1,0,3,2,7,1,5,3,4,4]
Dgenotype =  [0,0,1,0,2,1,1,0,3,2,7,1,5,3,4,4]
num_Slayer = 4
num_Dlayer=4
num_DDlayer=2
num_classes = 2

In [ ]:
ori_model=model.MftNetwork( C , Sgenotype, Dgenotype,num_Slayer, num_Dlayer, num_DDlayer ,num_classes)
print(ori_model)

In [ ]:
Model_optimizer = torch.optim.Adam(ori_model.parameters(),lr=lr)
num_batches = len(train_loader)
kl_loss = torch.nn.KLDivLoss(reduction = 'batchmean')
cls_loss = torch.nn.CrossEntropyLoss()
num_steps = num_epochs*num_batches
hist = np.zeros((num_steps,3))
index_i = -1

In [ ]:
def fa_train(Model,num_epochs,train_loader,lr,kl_loss,cls_loss,num_steps,hist,index_i,alpha,bate,zeta,delat,gama):
    Model_optimizer = torch.optim.Adam(Model.parameters(),lr=lr)
    Model.train()
    for epoch in range(num_epochs):
        for batch_index, src_data in enumerate(train_loader):
            index_i += 1
            Model_optimizer.zero_grad()
            X_train, Y_train, _ = src_data
            X_train = X_train.cuda()
            Y_train = Y_train.cuda()
            f1,f2,output = Model(X_train)
            _, src_prd_label = torch.max(output, 1) 
            
            Y_count=[]
            y_kl_surface = torch.tensor(0.0,requires_grad=True)
            y_kl_deep = torch.tensor(0.0,requires_grad=True)
            for y in Y_train:
                if y in Y_count: continue
                SC=[]
                for i in range(len(Y_train)):
                    if Y_train[i] == y:
                        SC.append(i)  
                if len(SC) > 1 :
                    y_class_kl_surface = class_kl_comput(f1,SC)
                    y_class_kl_deep = class_kl_comput(f2,SC)
                    y_kl_surface = y_kl_surface + y_class_kl_surface
                    y_kl_deep = y_kl_deep + y_class_kl_deep
                    Y_count.append(y)       
            if len(Y_count) != 0:
                y_kl_surface = y_kl_surface/len(Y_count)
                y_kl_deep = y_kl_deep/len(Y_count)
            Y_count=[]
            DC=[]
            for i in range(len(Y_train)):
                if Y_train[i] in Y_count: continue
                Y_count.append(Y_train[i])
                DC.append(i)         
            if len(Y_count) > 1:
                y_class_kl_other_surface = class_kl_comput(f1,DC)
                y_class_kl_other_deep = class_kl_comput(f2,DC)
            
            cls_loss_value = cls_loss(output, Y_train)
            cls_loss_value.requires_grad_(True)
            y_kl_surface.requires_grad_(True)
            y_kl_deep.requires_grad_(True)
            y_class_kl_other_surface.requires_grad_(True)
            y_class_kl_other_deep.requires_grad_(True)
            sum_loss= alpha*cls_loss_value + bate * y_kl_surface + zeta * y_kl_deep - delat * y_class_kl_other_surface - gama * y_class_kl_other_deep
            sum_loss.backward()
            Model_optimizer.step() 


    OA_new,_ = test_acc(Model,classname, val_loader, epoch+1,num_classes,print_per_batches=10)
    model_name = 'epoch_'+str(epoch+1)+'_OA_'+repr(int(OA_new*10000))+'.pth'
    save_path = os.path.join(save_path_prefix, model_name)
    model_count=1
    while os.path.exists(save_path)==True:
        model_name = 'epoch_'+str(epoch+1)+'_OA_'+repr(int(OA_new*10000))+ '_' + str(model_count) +'.pth'
        save_path = os.path.join(save_path_prefix, model_name)    
        model_count = model_count + 1
    
    save_path_new = os.path.join(save_path_prefix, model_name)    
    torch.save(Model.state_dict(), save_path_new)
    
    return save_path

In [ ]:
def cw_atk(Model,imloader):
    attack_func='cwlow'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    num_iter = 5
    C=50
    std = 0.1
    scale = 10
    beta = 1e-3
    epsilon = 0.1
    alpha = 1
    crop_size = 32
    for batch_index, src_data in enumerate(imloader):
                X, Y, img_name = src_data
                X = X.cuda()
                label_mask = F.one_hot(Y, num_classes=num_classes).cuda()    
                adv_im = X.clone().cuda()
                label = Y.clone().cuda()
                Y = Y.numpy().squeeze()            
                for i in range(num_iter):
                    adv_im.requires_grad = True     
                    _,_,out = Model(adv_im)
                
                    correct_logit = torch.sum(label_mask * out)     
                    wrong_logit = torch.max((1 - label_mask) * out)
                    loss = -(correct_logit - wrong_logit + C)       

                    grad = torch.autograd.grad(loss, adv_im,
                                            retain_graph=False, create_graph=False)[0]


                    adv_im = adv_im.detach() + alpha*grad / torch.norm(grad,float('inf'))
                    delta = torch.clamp(adv_im - X, min=-epsilon, max=epsilon)
                    adv_im = (X + delta).detach()
                
                    recreated_image = recreate_image(adv_im.cpu())
                    adv_im = preprocess_image_simple(Image.fromarray(recreated_image),crop_size)
            
                gen_name = img_name[0]+'_adv.png'
                im = Image.fromarray(recreated_image)
                im.save(adv_path_prefix+gen_name,'png')
                
                
def jitter_atk(Model,imloader):
    attack_func='jitterlow'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    num_iter = 5
    mse_loss = torch.nn.MSELoss(reduction='none')
    C=50
    std = 0.1
    scale = 10
    beta = 1e-3
    epsilon = 0.1
    alpha = 1
    crop_size = 32

    for batch_index, src_data in enumerate(imloader):       
        X, Y, img_name = src_data
        X = X.cuda()
        label_mask = F.one_hot(Y, num_classes=num_classes).cuda().float()  
        adv_im = X.clone().cuda()
        label = Y.clone().cuda()
        Y = Y.numpy().squeeze()

        # Start iteration
        for i in range(num_iter):
            adv_im.requires_grad = True     
            _,_,out = Model(adv_im)
            _, pre = torch.max(out, dim=1)
            wrong = (pre != label)

            norm_z = torch.norm(out, p=float('inf'), dim=1, keepdim=True)
            hat_z = nn.Softmax(dim=1)(scale*out/norm_z)

            hat_z = hat_z + std*torch.randn_like(hat_z)

            loss = mse_loss(hat_z, label_mask).mean(dim=1)

            norm_r = torch.norm((adv_im - X), p=float('inf'), dim=[1,2,3])
            nonzero_r = (norm_r != 0)
            loss[wrong*nonzero_r] /= norm_r[wrong*nonzero_r]

            loss = loss.mean()

            grad = torch.autograd.grad(loss, adv_im,
                                    retain_graph=False, create_graph=False)[0]


            adv_im = adv_im.detach() + alpha*grad / torch.norm(grad,float('inf'))
            delta = torch.clamp(adv_im - X, min=-epsilon, max=epsilon)
            adv_im = (X + delta).detach()
                
            recreated_image = recreate_image(adv_im.cpu())

        gen_name = img_name[0]+'_adv.png'
        im = Image.fromarray(recreated_image)
        im.save(adv_path_prefix+gen_name,'png')     
        
        
def fgsm_atk(Model,imloader):
    attack_func='fgsmlow'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    num_iter = 5
    mse_loss = torch.nn.MSELoss(reduction='none')
    C=50
    std = 0.1
    scale = 10
    beta = 1e-3
    epsilon = 0.1
    alpha = 1
    crop_size = 32

    for batch_index, src_data in enumerate(imloader):       
        X, Y, img_name = src_data
        X = X.cuda()
        adv_im = X.clone().cuda()
        label = Y.clone().cuda()
        Y = Y.numpy().squeeze()            

        adv_im.requires_grad = True     
        _,_,out = Model(adv_im)
        pred_loss = cls_loss(out, label)
        grad = torch.autograd.grad(pred_loss, adv_im,
                                retain_graph=False, create_graph=False)[0]

        adv_im = adv_im.detach() + epsilon*grad / torch.norm(grad,float('inf'))
        delta = torch.clamp(adv_im - X, min=-epsilon, max=epsilon)
        adv_im = (X + delta).detach()
            
        recreated_image = recreate_image(adv_im.cpu())        
        gen_name = img_name[0]+'_adv.png'
        im = Image.fromarray(recreated_image)
        im.save(adv_path_prefix+gen_name,'png')     

In [ ]:
import copy

def PGD(model, X, y, epsilon, niters=15, alpha=16/255):
    X = X.cuda()
    y = y.cuda()
    _,_,out  = model(X)
    ce = torch.nn.CrossEntropyLoss()(out, y)
    err = (out.data.max(1)[1] != y.data).float().sum() / X.size(0)

    X_pgd = Variable(X.data, requires_grad=True)
    for i in range(niters):
        opt = torch.optim.RMSprop([X_pgd], lr=0.1)
        opt.zero_grad()
        _,_,pgd_out=model(X_pgd)
        loss = torch.nn.CrossEntropyLoss()(pgd_out, y)
        loss.backward()
        eta = alpha * X_pgd.grad.data.sign()  
        X_pgd = Variable(X_pgd.data + eta, requires_grad=True)
        eta = torch.clamp(X_pgd.data - X.data, -epsilon, epsilon)
        X_pgd = Variable(X.data + eta, requires_grad=True)

    return X_pgd


def deepfool(images, net ,num_classes=2, overshoot=0.02, max_iter=50):

    """
       :param image: Image of size HxWx3
       :param net: network (input: images, output: values of activation **BEFORE** softmax).
       :param num_classes: num_classes (limits the number of classes to test against, by default = 10)
       :param overshoot: used as a termination criterion to prevent vanishing updates (default = 0.02).
       :param max_iter: maximum number of iterations for deepfool (default = 50)
       :return: minimal perturbation that fools the classifier, number of iterations that it required, new estimated_label and perturbed image
    """
    adv_image=[]

    for i in range(0, images.size()[0]):
        image=images[i]
        if image.shape == (3,32,32) :
            image = image.resize(1,3,32,32)
        image = image.cuda()
        net = net.cuda()     
        _,_,img_out=net.forward(Variable(image, requires_grad=True))
        f_image = img_out.data.cpu().numpy().flatten()
        I = (np.array(f_image)).flatten().argsort()[::-1]
        I = I[0:num_classes]
        label = I[0]
        input_shape = image.cpu().numpy().shape
        pert_image = copy.deepcopy(image)
        w = np.zeros(input_shape)
        r_tot = np.zeros(input_shape)
        loop_i = 0
        x = Variable(pert_image, requires_grad=True)
    
        _,_,fs = net.forward(x)
        fs_list = [fs[0,I[k]] for k in range(num_classes)]
        k_i = label

        while k_i == label and loop_i < max_iter:

            pert = np.inf
            fs[0, I[0]].backward(retain_graph=True)
            grad_orig = x.grad.data.cpu().numpy().copy()

            for k in range(1, num_classes):
                x.grad.zero_()
                fs[0, I[k]].backward(retain_graph=True)
                cur_grad = x.grad.data.cpu().numpy().copy()
                w_k = cur_grad - grad_orig
                f_k = (fs[0, I[k]] - fs[0, I[0]]).data.cpu().numpy()
                pert_k = abs(f_k)/np.linalg.norm(w_k.flatten())
                if pert_k < pert:
                    pert = pert_k
                    w = w_k
            r_i =  (pert+1e-4) * w / np.linalg.norm(w)
            r_tot = np.float32(r_tot + r_i)

            
            pert_image = image + (1+overshoot)*torch.from_numpy(r_tot).cuda()
            

            x = Variable(pert_image, requires_grad=True)
            _,_,fs = net.forward(x)
            k_i = np.argmax(fs.data.cpu().numpy().flatten())

            loop_i += 1

        r_tot = (1+overshoot)*r_tot
        adv_image.append(pert_image.resize(3,32,32))
        
    adv_image=torch.tensor([item.cpu().detach().numpy() for item in adv_image]).cuda()
    return adv_image


def noise(x, eps=0.1, order=np.inf, clip_min=None, clip_max=None):
    if order != np.inf:
        raise NotImplementedError(norm)

    eta = torch.FloatTensor(*x.shape).to(x.device).uniform_(-eps, eps)
    adv_x = x + eta

    if clip_min is not None or clip_max is not None:
        assert clip_min is not None and clip_max is not None
        adv_x = torch.clamp(adv_x, min=clip_min, max=clip_max)

    return adv_x

def pgd_atk(Model,imloader):
    attack_func='pgdlow'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    for batch_index, src_data in enumerate(imloader):
        X, Y, img_name = src_data
        eps=0.1
        adv=PGD(Model, X, Y, eps)
        recreated_image = recreate_image(adv.cpu()) 
        gen_name = img_name[0]+'_adv.png'
        im = Image.fromarray(recreated_image)
        im.save(adv_path_prefix+gen_name,'png') 
        
        
def deepfool_atk(Model,imloader):
    attack_func='deepfoollow'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    for batch_index, src_data in enumerate(imloader):
        X, Y, img_name = src_data
        eps=0.1
        adv=deepfool(X,Model)
        recreated_image = recreate_image(adv.cpu()) 
        gen_name = img_name[0]+'_adv.png'
        im = Image.fromarray(recreated_image)
        im.save(adv_path_prefix+gen_name,'png') 
        
def noise_atk(Model,imloader):
    attack_func='Noisy'
    surrogate_model='fnas'
    adv_dir = './'
    adv_path_prefix = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'
    if os.path.exists(adv_path_prefix)==False:
        os.makedirs(adv_path_prefix)
    for batch_index, src_data in enumerate(imloader):
        X, Y, img_name = src_data
        eps=0.1
        adv=noise(X)
        recreated_image = recreate_image(adv.cpu()) 
        gen_name = img_name[0]+'_adv.png'
        im = Image.fromarray(recreated_image)
        im.save(adv_path_prefix+gen_name,'png') 

In [ ]:
import random
from pymoo.core.problem import ElementwiseProblem
class MyProblem(ElementwiseProblem):
    def __init__(self,DataName,num_classes,classname,print_per_batches,val_loader,im_loader,
                composed_transforms,save_path_prefix,clean_loader,network,num_epochs):
        self.DataName = DataName
        self.num_classes = num_classes
        self.classname = classname
        self.print_per_batches = print_per_batches
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.im_loader = im_loader
        self.composed_transforms = composed_transforms
        self.save_path_prefix = save_path_prefix
        self.clean_loader = clean_loader
        self.network = network
        self.num_epochs = num_epochs
        super().__init__(n_var=1 + 6*2 +6*2+ 3+6, 
                         n_obj=3, 
                         n_constr=0, 
                         xl= np.array([4.0,
                                       0.0,0.0,0.0,0.0,0.0,0.0,
                                       0.0,0.0,0.0,0.0,0.0,0.0,
                                       0.0,0.0,0.0,0.0,0.0,0.0,
                                       0.0,0.0,0.0,0.0,0.0,0.0,
                                       1.0,
                                       1.0,
                                       1.0,
                             1e-4,0.9,0.01,0.01,0.01,0.01]), 
                         xu= np.array([32.4,
                                       0.4,1.0,2.0,3.0,4.0,5.0,
                                       12.0,12.0,12.0,12.0,12.0,12.0,
                                       0.4,1.0,2.0,3.0,4.0,5.0,
                                       12.0,12.0,12.0,12.0,12.0,12.0,
                                       4.4,
                                       4.4,
                                       4.4,
                             0.01,1.0,1.0,1.0,1.0,1.0]), 
                         type_var=float)
        

    def _evaluate(self, x, out, *args, **kwargs):

        C = x[0]
        Sgenotype = x[1:13]
        Dgenotype = x[13:25]
        C=model_nas_count(C)
        Sgenotype=model_nas_count(Sgenotype)
        Dgenotype=model_nas_count(Dgenotype)
        Slayer_num=x[25]
        Dlayer_num=x[26]
        DDlayer_num=x[27]
        Slayer_num=model_nas_count(Slayer_num)
        Dlayer_num=model_nas_count(Dlayer_num)
        DDlayer_num=model_nas_count(DDlayer_num)
        num_classes = 2
        DataName = 'OpenSARShip'
        ori_model=model.MftNetwork( C , Sgenotype, Dgenotype,Slayer_num,Dlayer_num,DDlayer_num,num_classes) 
        ori_model = ori_model.cuda()
        train_loader = data.DataLoader(
        scene_dataset(root_dir=root_dir,pathfile='./dataset/'+DataName+'_train_2_ex.txt', transform=composed_transforms),
        batch_size=train_batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
        num_batches = len(train_loader)
        kl_loss = torch.nn.KLDivLoss()
        cls_loss = torch.nn.CrossEntropyLoss()
        num_steps = self.num_epochs * num_batches
        hist = np.zeros((num_steps,3))
        index_i = -1   
        try:
            target_Model = fa_train(ori_model,self.num_epochs,
                         train_loader,x[28],kl_loss,
                         cls_loss,num_steps,hist,
                         index_i,x[29],x[30],x[31],
                         x[32],x[33])
            atk_choose = random.randrange(1, 7)
            if atk_choose == 1:
                fgsm_atk(ori_model,imloader)
                attack_func='fgsmlow'
            elif atk_choose == 2:
                pgd_atk(ori_model,imloader)
                attack_func='pgdlow'
            elif atk_choose == 3:
                cw_atk(ori_model,imloader)
                attack_func='cwlow'
            elif atk_choose == 4:
                deepfool_atk(ori_model,imloader)
                attack_func='deepfoollow'
            elif atk_choose == 5:
                jitter_atk(ori_model,imloader)
                attack_func='jitterlow'
            elif atk_choose == 6:
                noise_atk(ori_model,imloader)
                attack_func='Noisy'
            
            DataName='OpenSARShip'
            surrogate_model='fnas_2_0421'
            val_batch_size = 64
            adv_dir = './'
            adv_root_dir = adv_dir+DataName+'_adv/'+attack_func+'/'+surrogate_model+'/'    
            adv_loader = data.DataLoader(scene_dataset(root_dir=adv_root_dir,pathfile='./dataset/'+DataName+'_test_2_ex.txt', transform=composed_transforms, mode='adv'),
            batch_size=val_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
            OA_clean,_ = test_acc(ori_model,self.classname, self.clean_loader, 1,self.num_classes,self.print_per_batches)
            OA_adv,_ = test_acc(ori_model,self.classname, adv_loader, 1,self.num_classes,self.print_per_batches)
            num_params = sum(param.numel() for param in ori_model.parameters())
            f1 = -OA_clean 
            f2 = -OA_adv
            f3 = num_params
        
        except:
            print('bad network')
            num_params = sum(param.numel() for param in ori_model.parameters())
            f1 = 0.0
            f2 = 0.0
            f3 = num_params
            
            del ori_model
            torch.cuda.empty_cache()
            torch.cuda.empty_cache()
            time.sleep(5)

        out["F"] = [f1, f2, f3]
    

In [ ]:
(DataName,
 num_classes,
 classname,
 print_per_batches,
 train_loader, 
 val_loader,
 composed_transforms,
 save_path_prefix) = get_data(dataID,
                                print_per_batches,
                                save_path,
                                network,
                                crop_size,
                                root_dir,
                                train_batch_size,
                                val_batch_size,
                                num_workers)


num_epochs = 20
surrogate_model='fnas'
val_batch_size = 64
adv_dir = './'

clean_loader = data.DataLoader(
        scene_dataset(root_dir=root_dir,pathfile='./dataset/'+DataName+'_test_2_ex.txt', transform=composed_transforms),
        batch_size=val_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

imloader = data.DataLoader(
        scene_dataset(root_dir=root_dir,pathfile='./dataset/'+DataName+'_test_2_ex.txt', transform=composed_transforms),
        batch_size=1, shuffle=True, num_workers=num_workers, pin_memory=True)


In [ ]:
problem = MyProblem(DataName,num_classes,classname,print_per_batches,val_loader,imloader,
                   composed_transforms,save_path_prefix,clean_loader,network,num_epochs)


In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.factory import get_sampling, get_crossover, get_mutation

algorithm = NSGA2(
    pop_size=20,
    n_offsprings=20,
    sampling=get_sampling("real_random"),
    crossover=get_crossover("real_sbx", prob=0.9, eta=15),
    mutation=get_mutation("real_pm", eta=20),
    eliminate_duplicates=True
)

In [ ]:
from pymoo.factory import get_termination
from pymoo.optimize import minimize

termination = get_termination("n_gen", 20)

res = minimize(
    problem,
    algorithm,
    termination,
    seed=4,
    verbose=True, 
    save_history=True 
)
X = res.X
F = res.F

In [ ]:
print(res.X)
print(res.F)

In [ ]:
np.savetxt('mftnetwork_results_opensar_2_240421_x_seed4.txt',res.X)
np.savetxt('mftnetwork_results_opensar_2_240421_y_seed4.txt',res.F)